# CS781 Capstone - Phase 5: Legal Statute Identification (LSI), multi-label
Name: Aditya Kumar | Roll No: 240059 | Email: adityaj24@iitk.ac.in

In [31]:
import os, re, gc, glob, math, json, time, random, shutil, zipfile, hashlib, subprocess, sys, csv
import numpy as np
import torch
import torch.nn as nn
import torch.nn.functional as F

DEVICE = torch.device("cuda" if torch.cuda.is_available() else "cpu")
NUM_GPUS = torch.cuda.device_count()
print(DEVICE, NUM_GPUS)

cuda 2


In [32]:
                                                                          
ARCH = "encdec"                                                                                                                       
RUNS = None                                                                                                                                                                                                                          
DEFAULT_RUNS = {
    "decoder":     [dict(method="qlora"),   dict(method="lorma")],
    "encdec":      [dict(method="adapter"), dict(method="partial")],
    "hybrid":      [dict(method="full"),    dict(method="lorma")],
    "lightweight": [dict(method="adapter"), dict(method="kd")],                                                                                    
    "rope_qknorm": [dict(method="full"),    dict(method="adapter")],
}
assert ARCH in DEFAULT_RUNS, f"ARCH must be one of {list(DEFAULT_RUNS)}"
ENSEMBLE = True                                                             
ENSEMBLE_SCOPE = "arch"                                                                                                                                               
EXP_TAG = "p5"
SMOKE = False                                                                                                            

                                                                     
hf_token = os.environ.get("HF_TOKEN")                                               
HF_REPO, P5_DIR = "Exploration-Lab/CS-781-Capstone", "Phase_5_LSI"
LOCAL_DATA_DIR = None
INPUT_ROOT = os.environ.get("CS781_INPUT_ROOT", "/kaggle/input")
OUT_ROOT = os.environ.get("CS781_OUT_ROOT", "/kaggle/working")

                                                                               
                                                                                            
CHUNK_WORDS = 200                                                                                
WINDOW = 512                                                        
K_CHUNKS = 4                                                                                       
DOC_MODE = "tail"                                                                                                                        
POOL = "attn"                                                                                           
USE_POS_FEATURE = True                                                                  
BIDIRECTIONAL = True                                                             

                                                                        
EPOCHS = {"full": 4, "partial": 5, "bitfit": 6, "lora": 5, "lorma": 5, "adapter": 5, "qlora": 5, "kd": 5}
LR     = {"full": 5e-5, "partial": 1e-4, "bitfit": 1e-3, "lora": 1e-3, "lorma": 5e-4, "adapter": 5e-4, "qlora": 1e-3, "kd": 1e-4}
HEAD_LR, WEIGHT_DECAY, WARMUP_FRAC, PATIENCE = 2e-3, 0.01, 0.06, 2
BATCH_DOCS, GRAD_ACCUM = 4, 4                                                                                                                                              
POS_WEIGHT_MAX = 20.0                                                                                                                                          
HEAD_HIDDEN = 512
THRESH_GRID = [round(x, 3) for x in np.arange(0.05, 0.96, 0.025)]                                                                
ENSURE_ONE = True                                                                                                  
TRAIN_DOC_LIMIT = None                                                                        
TIME_BUDGET_MIN = 300                                                                                                                                                         
RESERVE_MIN = 25                                                                
OVERHEAD_MIN = 4                                                                                   
MIN_RUN_MIN = 20                                                                                                                   
SESSION_T0 = time.time()
RUN_MAX_MIN = None                                                                                 
PEFT = dict(r=16, alpha=32, targets=("q_proj", "k_proj", "v_proj", "out_proj"), lorma_targets=("q_proj", "v_proj"), bottleneck=64, partial_last=4)
KD_T, KD_ALPHA, STUDENT_KEEP_EVERY = 2.0, 0.5, 2

SEED = 42
random.seed(SEED); np.random.seed(SEED); torch.manual_seed(SEED)
print("arch:", ARCH, "| runs:", [r["method"] for r in (RUNS or DEFAULT_RUNS[ARCH])])

arch: encdec | runs: ['adapter', 'partial']


In [33]:
ROPE_BASE = 10000.0

                                                  
class CausalSelfAttention(nn.Module):
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class GPTBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = CausalSelfAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x):
        x = x + self.attn(self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class DecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([GPTBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            logits, _ = self(idx[:, -self.max_seq_len:])
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

                                                                    
class MultiHeadAttention(nn.Module):
    """Generic multi-head attention. causal=True masks future query->key positions (self-attn
    in the decoder). query and key/value can come from different sequences (cross-attention)."""
    def __init__(self, d_model, n_heads, dropout=0.0, causal=False, max_seq_len=None):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.causal = causal
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        if causal:
            mask = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
            self.register_buffer("causal_mask", mask, persistent=False)

    def forward(self, x_q, x_kv):
        B, T, C = x_q.shape
        S = x_kv.shape[1]
        q = self.q_proj(x_q).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x_kv).view(B, S, self.n_heads, self.d_head).transpose(1, 2)
        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        if self.causal:
            scores = scores.masked_fill(~self.causal_mask[:T, :S], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


def feed_forward(d_model, d_ff, dropout):
    return nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))


class EncoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.ff(self.ln2(x))
        return x


class DecoderBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_tgt_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.self_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=True, max_seq_len=max_tgt_len)
        self.ln2 = nn.LayerNorm(d_model)
        self.cross_attn = MultiHeadAttention(d_model, n_heads, dropout, causal=False)
        self.ln3 = nn.LayerNorm(d_model)
        self.ff = feed_forward(d_model, d_ff, dropout)

    def forward(self, x, enc_out):
        x = x + self.self_attn(self.ln1(x), self.ln1(x))
        x = x + self.cross_attn(self.ln2(x), enc_out)
        x = x + self.ff(self.ln3(x))
        return x


class EncoderDecoderLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_enc_layers, n_dec_layers, n_heads, d_ff,
                 max_src_len, max_tgt_len, dropout=0.1):
        super().__init__()
        self.max_src_len = max_src_len
        self.max_tgt_len = max_tgt_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("enc_pos", sinusoidal_positional_encoding(max_src_len, d_model), persistent=False)
        self.register_buffer("dec_pos", sinusoidal_positional_encoding(max_tgt_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.encoder_blocks = nn.ModuleList([EncoderBlock(d_model, n_heads, d_ff, dropout) for _ in range(n_enc_layers)])
        self.decoder_blocks = nn.ModuleList([DecoderBlock(d_model, n_heads, d_ff, max_tgt_len, dropout) for _ in range(n_dec_layers)])
        self.enc_ln_f = nn.LayerNorm(d_model)
        self.dec_ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def encode(self, src_ids):
        T = src_ids.shape[1]
        x = self.drop(self.token_emb(src_ids) + self.enc_pos[:T].unsqueeze(0))
        for block in self.encoder_blocks:
            x = block(x)
        return self.enc_ln_f(x)

    def decode(self, tgt_input_ids, enc_out):
        T = tgt_input_ids.shape[1]
        x = self.drop(self.token_emb(tgt_input_ids) + self.dec_pos[:T].unsqueeze(0))
        for block in self.decoder_blocks:
            x = block(x, enc_out)
        return self.lm_head(self.dec_ln_f(x))

    def forward(self, src_ids, tgt_input_ids, tgt_target_ids=None):
        enc_out = self.encode(src_ids)
        logits = self.decode(tgt_input_ids, enc_out)
        loss = None
        if tgt_target_ids is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), tgt_target_ids.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, src_ids, bos_id, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        enc_out = self.encode(src_ids[:, -self.max_src_len:])
        dec_ids = torch.full((src_ids.size(0), 1), bos_id, dtype=torch.long, device=src_ids.device)
        for _ in range(max_new_tokens):
            logits = self.decode(dec_ids[:, -self.max_tgt_len:], enc_out)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=src_ids.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            dec_ids = torch.cat([dec_ids, next_token], dim=1)
        return dec_ids[:, 1:]

                                                      
class PrefixAwareAttention(nn.Module):
    """Causal self-attention, except query positions inside the prefix (index < prefix_len)
    also get bidirectional access to the rest of the prefix. Positions >= prefix_len are
    plain causal. A single stack, single attention type per layer - neither a fully causal
    decoder nor a two-stack encoder-decoder."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, prefix_len):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        pos = torch.arange(T, device=x.device)
        in_prefix = pos < prefix_len
        prefix_bonus = in_prefix.unsqueeze(1) & in_prefix.unsqueeze(0)
        allowed = self.causal_mask[:T, :T] | prefix_bonus

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~allowed, float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


def sinusoidal_positional_encoding(max_len, d_model):
    pe = torch.zeros(max_len, d_model)
    position = torch.arange(0, max_len, dtype=torch.float32).unsqueeze(1)
    div_term = torch.exp(torch.arange(0, d_model, 2, dtype=torch.float32) * (-math.log(10000.0) / d_model))
    pe[:, 0::2] = torch.sin(position * div_term)
    pe[:, 1::2] = torch.cos(position * div_term)
    return pe


class PrefixLMBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.ln1 = nn.LayerNorm(d_model)
        self.attn = PrefixAwareAttention(d_model, n_heads, max_seq_len, dropout)
        self.ln2 = nn.LayerNorm(d_model)
        self.ff = nn.Sequential(nn.Linear(d_model, d_ff), nn.GELU(), nn.Linear(d_ff, d_model), nn.Dropout(dropout))

    def forward(self, x, prefix_len):
        x = x + self.attn(self.ln1(x), prefix_len)
        x = x + self.ff(self.ln2(x))
        return x


class PrefixLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.register_buffer("pos_emb", sinusoidal_positional_encoding(max_seq_len, d_model), persistent=False)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([PrefixLMBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.ln_f = nn.LayerNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, prefix_len, targets=None, loss_mask=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx) + self.pos_emb[:T].unsqueeze(0))
        for block in self.blocks:
            x = block(x, prefix_len)
        logits = self.lm_head(self.ln_f(x))
        loss = None
        if targets is not None:
            per_tok = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.reshape(-1), reduction="none").view(B, T)
            m = (torch.arange(T, device=idx.device) + 1 >= prefix_len).to(per_tok.dtype)
            loss = (per_tok * m).sum() / (m.sum() * B).clamp(min=1.0)
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, prefix_len, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond, prefix_len=min(prefix_len, idx_cond.shape[1]))
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

                                                                           
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight


def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
                                                        
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class GQAAttention(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, max_seq_len, dropout=0.0):
        super().__init__()
        assert n_heads % n_kv_heads == 0
        self.n_heads = n_heads
        self.n_kv_heads = n_kv_heads
        self.n_rep = n_heads // n_kv_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, n_heads * self.d_head, bias=False)
        self.k_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.v_proj = nn.Linear(d_model, n_kv_heads * self.d_head, bias=False)
        self.out_proj = nn.Linear(n_heads * self.d_head, d_model, bias=False)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_kv_heads, self.d_head).transpose(1, 2)

        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        k = k.repeat_interleave(self.n_rep, dim=1)
        v = v.repeat_interleave(self.n_rep, dim=1)

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class LightBlock(nn.Module):
    def __init__(self, d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = GQAAttention(d_model, n_heads, n_kv_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class LightweightLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, n_kv_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([
            LightBlock(d_model, n_heads, n_kv_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)
        ])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

                                                                       
class RMSNorm(nn.Module):
    def __init__(self, dim, eps=1e-6):
        super().__init__()
        self.eps = eps
        self.weight = nn.Parameter(torch.ones(dim))

    def forward(self, x):
        norm = x * torch.rsqrt(x.pow(2).mean(dim=-1, keepdim=True) + self.eps)
        return norm * self.weight


def precompute_rope(d_head, max_seq_len, base=ROPE_BASE):
    inv_freq = 1.0 / (base ** (torch.arange(0, d_head, 2).float() / d_head))
    t = torch.arange(max_seq_len).float()
    freqs = torch.outer(t, inv_freq)
    return torch.cos(freqs), torch.sin(freqs)


def apply_rope(x, cos, sin):
    d = x.shape[-1]
    x1, x2 = x[..., : d // 2], x[..., d // 2 :]
    cos = cos.unsqueeze(0).unsqueeze(0)
    sin = sin.unsqueeze(0).unsqueeze(0)
    return torch.cat([x1 * cos - x2 * sin, x2 * cos + x1 * sin], dim=-1)


class SwiGLU(nn.Module):
    def __init__(self, d_model, d_ff, dropout):
        super().__init__()
        self.gate_proj = nn.Linear(d_model, d_ff, bias=False)
        self.up_proj = nn.Linear(d_model, d_ff, bias=False)
        self.down_proj = nn.Linear(d_ff, d_model, bias=False)
        self.dropout = nn.Dropout(dropout)

    def forward(self, x):
        return self.dropout(self.down_proj(F.silu(self.gate_proj(x)) * self.up_proj(x)))


class QKNormAttention(nn.Module):
    """Full multi-head causal attention with RoPE, plus QK-Norm: Q and K are each RMSNorm'd
    per head (over the head dimension) before the dot product, bounding attention logit scale
    independent of how large Q/K activations get during training - improves stability."""
    def __init__(self, d_model, n_heads, max_seq_len, dropout=0.0):
        super().__init__()
        self.n_heads = n_heads
        self.d_head = d_model // n_heads
        self.scale = 1.0 / math.sqrt(self.d_head)
        self.q_proj = nn.Linear(d_model, d_model, bias=False)
        self.k_proj = nn.Linear(d_model, d_model, bias=False)
        self.v_proj = nn.Linear(d_model, d_model, bias=False)
        self.out_proj = nn.Linear(d_model, d_model, bias=False)
        self.q_norm = RMSNorm(self.d_head)
        self.k_norm = RMSNorm(self.d_head)
        self.attn_dropout = nn.Dropout(dropout)
        self.resid_dropout = nn.Dropout(dropout)
        causal = torch.tril(torch.ones(max_seq_len, max_seq_len, dtype=torch.bool))
        self.register_buffer("causal_mask", causal, persistent=False)

    def forward(self, x, cos, sin):
        B, T, C = x.shape
        q = self.q_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        k = self.k_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)
        v = self.v_proj(x).view(B, T, self.n_heads, self.d_head).transpose(1, 2)

        q = self.q_norm(q)
        k = self.k_norm(k)
        q = apply_rope(q, cos[:T], sin[:T])
        k = apply_rope(k, cos[:T], sin[:T])

        scores = torch.matmul(q, k.transpose(-2, -1)) * self.scale
        scores = scores.masked_fill(~self.causal_mask[:T, :T], float("-inf"))
        attn = self.attn_dropout(torch.softmax(scores, dim=-1))
        out = torch.matmul(attn, v).transpose(1, 2).contiguous().view(B, T, C)
        return self.resid_dropout(self.out_proj(out))


class QKBlock(nn.Module):
    def __init__(self, d_model, n_heads, d_ff, max_seq_len, dropout):
        super().__init__()
        self.norm1 = RMSNorm(d_model)
        self.attn = QKNormAttention(d_model, n_heads, max_seq_len, dropout)
        self.norm2 = RMSNorm(d_model)
        self.ff = SwiGLU(d_model, d_ff, dropout)

    def forward(self, x, cos, sin):
        x = x + self.attn(self.norm1(x), cos, sin)
        x = x + self.ff(self.norm2(x))
        return x


class RopeSwiGLUQKNormLM(nn.Module):
    def __init__(self, vocab_size, d_model, n_layers, n_heads, d_ff, max_seq_len, dropout=0.1):
        super().__init__()
        self.max_seq_len = max_seq_len
        self.token_emb = nn.Embedding(vocab_size, d_model)
        self.drop = nn.Dropout(dropout)
        self.blocks = nn.ModuleList([QKBlock(d_model, n_heads, d_ff, max_seq_len, dropout) for _ in range(n_layers)])
        self.norm_f = RMSNorm(d_model)
        self.lm_head = nn.Linear(d_model, vocab_size, bias=False)
        self.lm_head.weight = self.token_emb.weight

        d_head = d_model // n_heads
        cos, sin = precompute_rope(d_head, max_seq_len)
        self.register_buffer("rope_cos", cos, persistent=False)
        self.register_buffer("rope_sin", sin, persistent=False)
        self.apply(self._init_weights)

    def _init_weights(self, module):
        if isinstance(module, (nn.Linear, nn.Embedding)):
            nn.init.normal_(module.weight, mean=0.0, std=0.02)
            if isinstance(module, nn.Linear) and module.bias is not None:
                nn.init.zeros_(module.bias)

    def forward(self, idx, targets=None):
        B, T = idx.shape
        x = self.drop(self.token_emb(idx))
        for block in self.blocks:
            x = block(x, self.rope_cos, self.rope_sin)
        logits = self.lm_head(self.norm_f(x))
        loss = None
        if targets is not None:
            loss = F.cross_entropy(logits.view(-1, logits.size(-1)), targets.view(-1))
        return logits, loss

    @torch.no_grad()
    def generate(self, idx, max_new_tokens, temperature=1.0, top_p=0.95):
        self.eval()
        for _ in range(max_new_tokens):
            idx_cond = idx[:, -self.max_seq_len:]
            logits, _ = self(idx_cond)
            logits = logits[:, -1, :] / max(temperature, 1e-5)
            probs = torch.softmax(logits, dim=-1)
            sorted_probs, sorted_idx = torch.sort(probs, descending=True, dim=-1)
            cum_probs = torch.cumsum(sorted_probs, dim=-1)
            cutoff = (cum_probs > top_p).float().argmax(dim=-1, keepdim=True)
            mask = torch.arange(sorted_probs.size(-1), device=idx.device).unsqueeze(0) > cutoff
            sorted_probs = sorted_probs.masked_fill(mask, 0.0)
            sorted_probs = sorted_probs / sorted_probs.sum(dim=-1, keepdim=True)
            next_sorted = torch.multinomial(sorted_probs, num_samples=1)
            next_token = torch.gather(sorted_idx, -1, next_sorted)
            idx = torch.cat([idx, next_token], dim=1)
        return idx

In [34]:
CAUSAL_ARCHS = ("decoder", "lightweight", "rope_qknorm")

def find_phase1(arch):
    hits = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_model.pt"), recursive=True) or glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}.pt"), recursive=True)
    assert hits, f"{arch}_model.pt (or {arch}.pt) not found under {INPUT_ROOT}. Attach the dataset made from your Phase-1 zip."
    d = os.path.dirname(hits[0])
    for n in (f"{arch}_bpe_tokenizer.json", "bpe_tokenizer.json"):
        if os.path.exists(os.path.join(d, n)): return hits[0], os.path.join(d, n)
    t = glob.glob(os.path.join(INPUT_ROOT, "**", f"{arch}_bpe_tokenizer.json"), recursive=True) or glob.glob(os.path.join(INPUT_ROOT, "**", "bpe_tokenizer.json"), recursive=True)
    assert t, "tokenizer json not found next to the model"
    return hits[0], t[0]

def load_phase1_backbone(arch, path):
    ck = torch.load(path, map_location="cpu", weights_only=False); cfg, sd = ck["config"], ck["state_dict"]; g = cfg.get
    if arch == "decoder": m = DecoderLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "encdec": m = EncoderDecoderLM(g("vocab_size"), g("d_model"), g("n_enc_layers"), g("n_dec_layers"), g("n_heads"), g("d_ff"), g("max_src_len"), g("max_tgt_len"), g("dropout", 0.1))
    elif arch == "hybrid": m = PrefixLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "lightweight": m = LightweightLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("n_kv_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    elif arch == "rope_qknorm": m = RopeSwiGLUQKNormLM(g("vocab_size"), g("d_model"), g("n_layers"), g("n_heads"), g("d_ff"), g("max_seq_len"), g("dropout", 0.1))
    else: raise ValueError(arch)
    m.load_state_dict(sd, strict=True)
    return m, cfg, (g("max_seq_len") or g("max_src_len"))

def block_list_name(arch): return "encoder_blocks" if arch == "encdec" else "blocks"

def strip_unused(m, arch):
    m.lm_head = nn.Identity()
    if arch == "encdec": m.decoder_blocks = nn.ModuleList(); m.dec_ln_f = nn.Identity()

def prune_layers(m, arch, keep):
    blocks = getattr(m, block_list_name(arch)); setattr(m, block_list_name(arch), nn.ModuleList([blocks[i] for i in keep]))

In [35]:
                                                                                                                       
class LoRALinear(nn.Module):
    """LoRA: y = Wx + s*B(Ax). rsLoRA = same with s = alpha/sqrt(r) instead of alpha/r (rank-stabilised)."""
    def __init__(self, base, r, alpha, rs=False):
        super().__init__()
        self.base, self.s = base, (alpha / math.sqrt(r) if rs else alpha / r)
        self.A = nn.Parameter(torch.empty(r, base.in_features)); self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def forward(self, x): return self.base(x) + self.s * F.linear(F.linear(x, self.A), self.B)

class LoRMALinear(nn.Module):
    """LoRMA+ : W' = (I + (alpha/r) B A) W0 (multiplicative); A=0, B=kaiming as in the authors' code, so W'=W0 at step 0."""
    def __init__(self, base, r, alpha):
        super().__init__()
        self.base, self.s = base, alpha / r
        self.A = nn.Parameter(torch.zeros(r, base.out_features)); self.B = nn.Parameter(torch.empty(base.out_features, r))
        nn.init.kaiming_uniform_(self.B, a=math.sqrt(5))
    def forward(self, x):
        h = F.linear(x, self.base.weight); y = h + self.s * F.linear(F.linear(h, self.A), self.B)
        return y + self.base.bias if self.base.bias is not None else y

In [36]:
class BPE:
    """Phase-1 byte-level BPE (load + encode). Same merges and special ids as pretraining."""
    SPECIAL = ["<pad>", "<unk>", "<s>", "</s>"]
    def __init__(self):
        self.merges = {}; self.id_to_bytes = {i: bytes([i]) for i in range(256)}; self.special_to_id = {}
        for i, t in enumerate(self.SPECIAL): self.special_to_id[t] = 256 + i; self.id_to_bytes[256 + i] = t.encode()
    @staticmethod
    def _merge(seq, pair, new_id):
        out, i = [], 0
        while i < len(seq):
            if i < len(seq) - 1 and (seq[i], seq[i + 1]) == pair: out.append(new_id); i += 2
            else: out.append(seq[i]); i += 1
        return out
    @classmethod
    def load(cls, path):
        tok = cls()
        for ps, nid in json.load(open(path, encoding="utf-8"))["merges"].items():
            a, b = (int(x) for x in ps.split(",")); tok.merges[(a, b)] = nid; tok.id_to_bytes[nid] = tok.id_to_bytes[a] + tok.id_to_bytes[b]
        return tok
    def encode_piece(self, piece):
        seq = list(piece.encode("utf-8"))
        while len(seq) > 1:
            cand = [p for p in zip(seq, seq[1:]) if p in self.merges]
            if not cand: break
            pair = min(cand, key=lambda p: self.merges[p]); seq = self._merge(seq, pair, self.merges[pair])
        return seq

class WordTok:
    def __init__(self, bpe): self.bpe, self.eos, self.pad, self.cache = bpe, bpe.special_to_id["</s>"], bpe.special_to_id["<pad>"], {}
    def encode_words(self, words, max_tok):
        ids = []
        for w in words:
            p = self.cache.get(w)
            if p is None: p = self.cache[w] = self.bpe.encode_piece(w) + [self.eos]
            ids.extend(p)
        return np.array((ids or [self.eos])[-max_tok:], dtype=np.int64)                                           

def read_json(p):
    with open(p, "r", encoding="utf-8") as f: return json.load(f)
def get_path(rel):
    if LOCAL_DATA_DIR: return os.path.join(LOCAL_DATA_DIR, rel)
    from huggingface_hub import hf_hub_download
    return hf_hub_download(repo_id=HF_REPO, filename=f"{P5_DIR}/{rel}", repo_type="dataset", token=hf_token)

def parse_cases(raw, labelled):
    """[{id,text,labels:[statute ids]}] or {id: {text,labels}}."""
    pairs = list(raw.items()) if isinstance(raw, dict) else [(r.get("id", r.get("case_id")), r) for r in raw]
    out = []
    for cid, v in pairs:
        if not isinstance(v, dict): v = {"text": v}
        text = next((v[k] for k in ("text", "document", "case_text", "content") if k in v), None)
        assert text is not None, f"case {cid}: no text field (keys: {list(v)})"
        if isinstance(text, list): text = " ".join(map(str, text))
        labs = None
        if labelled:
            k = next((k for k in ("labels", "label", "statutes", "statute_ids", "statute_id") if k in v), None)
            assert k is not None, f"case {cid}: no label field (keys: {list(v)})"
            raw_l = v[k] if isinstance(v[k], (list, tuple)) else [v[k]]
            labs = sorted({int(float(x)) for x in raw_l})
        out.append(dict(id=str(cid), text=str(text), label=labs))
    return sorted(out, key=lambda d: d["id"])                                                                       

DOCS = {"train": parse_cases(read_json(get_path("Train/train.json")), True),
        "dev": parse_cases(read_json(get_path("Train/dev.json")), True),
        "test": parse_cases(read_json(get_path("Test/test.json")), False)}
for sp, ds in DOCS.items(): assert len({d["id"] for d in ds}) == len(ds), f"duplicate ids in {sp}"
NL = 1 + max(l for sp in ("train", "dev") for d in DOCS[sp] for l in d["label"])                                                                                   
def multihot(docs):
    Y = np.zeros((len(docs), NL), np.int64)
    for i, d in enumerate(docs):
        for l in d["label"]: Y[i, l] = 1
    return Y
if TRAIN_DOC_LIMIT and len(DOCS["train"]) > TRAIN_DOC_LIMIT: DOCS["train"] = random.Random(SEED).sample(DOCS["train"], TRAIN_DOC_LIMIT)
if SMOKE:
    DOCS = {"train": DOCS["train"][:16], "dev": DOCS["dev"][:8], "test": DOCS["test"][:6]}; EPOCHS = {k: 1 for k in EPOCHS}; K_CHUNKS = 2
    print(">>> SMOKE MODE: tiny subset, 1 epoch, 2 chunks <<<")
for sp, ds in DOCS.items():
    wl = [len(d["text"].split()) for d in ds]
    lab = f" | statutes/case {np.mean([len(d['label']) for d in ds]):.2f}" if ds[0]["label"] is not None else ""
    print(f"{sp:5s}: {len(ds):6d} cases | words/case median {int(np.median(wl))}, max {max(wl)}{lab}")
_pos = multihot(DOCS["train"]).sum(0)
print(f"label space: {NL} statute ids | present in train: {int((_pos > 0).sum())} | rare (<5 train positives): {int(((_pos > 0) & (_pos < 5)).sum())} | never seen: {int((_pos == 0).sum())}")

def pick_chunks(nc, K, mode):
    if nc <= K: return list(range(nc))
    if mode == "tail": return list(range(nc - K, nc))
    if mode == "head": return list(range(K))
    if mode == "headtail": h = K // 2; return list(range(h)) + list(range(nc - (K - h), nc))
    if mode == "spread": return sorted(set(np.linspace(0, nc - 1, K).round().astype(int).tolist()))
    raise ValueError(mode)

def pack_docs(docs, wt, mode, K):
    N = len(docs); X = np.full((N, K, WINDOW), wt.pad, np.int16); M = np.zeros((N, K), bool); P = np.zeros((N, K), np.float32)
    y = multihot(docs).astype(np.float32) if docs[0]["label"] is not None else np.zeros((N, NL), np.float32)
    for i, d in enumerate(docs):
        words = d["text"].split(); nc = max(1, -(-len(words) // CHUNK_WORDS))
        for k, c in enumerate(pick_chunks(nc, K, mode)):
            ids = wt.encode_words(words[c * CHUNK_WORDS:(c + 1) * CHUNK_WORDS], WINDOW)
            X[i, k, :len(ids)] = ids; M[i, k] = True; P[i, k] = c / max(nc - 1, 1)
    return dict(X=torch.from_numpy(X), M=torch.from_numpy(M), P=torch.from_numpy(P), y=torch.from_numpy(y))

train:  42750 cases | words/case median 430, max 86858 | statutes/case 3.70
dev  :  10181 cases | words/case median 543, max 69230 | statutes/case 3.92
test :  13019 cases | words/case median 532, max 134848
label space: 100 statute ids | present in train: 100 | rare (<5 train positives): 0 | never seen: 0


In [37]:
CAUSAL_ARCHS = ("decoder", "lightweight", "rope_qknorm")

class DocModel(nn.Module):
    """Phase-1 backbone -> mean-pool tokens per chunk -> (attention) pool chunks per document -> one logit per statute (multi-label)."""
    def __init__(self, arch, backbone, d_model, pad_id, n_labels, bidirectional=True, pool="attn", use_pos=True):
        super().__init__()
        self.arch, self.backbone, self.bidir, self.pool, self.use_pos, self.pad_id, self.d_model = arch, backbone, bidirectional, pool, use_pos, pad_id, d_model
        self.att = nn.Linear(d_model + (1 if use_pos else 0), 1)
        self.head = nn.Sequential(nn.Dropout(0.2), nn.Linear(d_model, HEAD_HIDDEN), nn.GELU(), nn.Dropout(0.2), nn.Linear(HEAD_HIDDEN, n_labels))
        if bidirectional and arch in CAUSAL_ARCHS:
            for mod in backbone.modules():
                if hasattr(mod, "causal_mask"): mod.causal_mask.fill_(True)

    def token_features(self, ids):
        m = self.backbone; e = m.token_emb(ids); T = e.shape[1]
        if self.arch == "decoder":
            x = m.drop(e + m.pos_emb[:T].unsqueeze(0))
            for b in m.blocks: x = b(x)
            return m.ln_f(x)
        if self.arch == "encdec":
            x = m.drop(e + m.enc_pos[:T].unsqueeze(0))
            for b in m.encoder_blocks: x = b(x)
            return m.enc_ln_f(x)
        if self.arch == "hybrid":
            x = m.drop(e + m.pos_emb[:T].unsqueeze(0)); plen = T if self.bidir else 0
            for b in m.blocks: x = b(x, plen)
            return m.ln_f(x)
        x = m.drop(e)
        for b in m.blocks: x = b(x, m.rope_cos, m.rope_sin)
        return m.norm_f(x)

    def forward(self, X, M, P):
        B, K, T = X.shape; ids = X.reshape(B * K, T).long(); valid = M.reshape(-1)
        h = self.token_features(ids[valid]).float()                                                     
        tm = (ids[valid] != self.pad_id).float().unsqueeze(-1)
        vec = (h * tm).sum(1) / tm.sum(1).clamp(min=1)
        full = torch.zeros(B * K, self.d_model, device=vec.device, dtype=vec.dtype); full[valid] = vec; full = full.view(B, K, -1)
        if self.pool == "mean": w = M.float() / M.float().sum(1, keepdim=True).clamp(min=1)
        else:
            s = self.att(torch.cat([full, P.unsqueeze(-1).to(full.dtype)], -1) if self.use_pos else full).squeeze(-1)
            w = torch.softmax(s.float().masked_fill(~M, -1e9), 1)
        return self.head((w.unsqueeze(-1) * full).sum(1).to(self.head[1].weight.dtype))

In [38]:
NF4 = [-1.0, -0.6961928009986877, -0.5250730514526367, -0.39491748809814453, -0.28444138169288635, -0.18477343022823334,
       -0.09105003625154495, 0.0, 0.07958029955625534, 0.16093020141124725, 0.24611230194568634, 0.33791524171829224,
       0.44070982933044434, 0.5626170039176941, 0.7229568362236023, 1.0]

class QLoRALinear(nn.Module):
    """QLoRA: frozen base weight stored as block-wise 4-bit NF4 (two codes per byte, block=64, absmax scale) + trainable LoRA."""
    def __init__(self, base, r, alpha, block=64):
        super().__init__()
        w = base.weight.detach().float().flatten(); self.n, self.shape, self.block = w.numel(), tuple(base.weight.shape), block
        if (-self.n) % block: w = F.pad(w, (0, (-self.n) % block))
        blocks = w.view(-1, block); absmax = blocks.abs().max(1, keepdim=True).values.clamp(min=1e-8)
        table = torch.tensor(NF4, device=w.device)
        codes = ((blocks / absmax).unsqueeze(-1) - table).abs().argmin(-1).to(torch.uint8)
        self.register_buffer("packed", (codes[:, 0::2] << 4) | codes[:, 1::2]); self.register_buffer("absmax", absmax.squeeze(1))
        self.register_buffer("table", table)
        self.register_buffer("bias", None if base.bias is None else base.bias.detach().clone())
        self.s = alpha / r; self.in_features, self.out_features = base.in_features, base.out_features
        self.A = nn.Parameter(torch.empty(r, base.in_features)); self.B = nn.Parameter(torch.zeros(base.out_features, r))
        nn.init.kaiming_uniform_(self.A, a=math.sqrt(5))
    def dequant(self):
        codes = torch.stack([(self.packed >> 4).long(), (self.packed & 0xF).long()], -1).view(self.packed.shape[0], -1)
        return (self.table[codes] * self.absmax[:, None]).flatten()[:self.n].view(self.shape)
    def forward(self, x):
        return F.linear(x, self.dequant().to(x.dtype), None if self.bias is None else self.bias.to(x.dtype)) + self.s * F.linear(F.linear(x, self.A), self.B)

class Adapter(nn.Module):
    """Houlsby bottleneck adapter; up-projection starts at zero so the model starts identical to Phase 1."""
    def __init__(self, d, bottleneck):
        super().__init__(); self.down, self.up, self.act = nn.Linear(d, bottleneck), nn.Linear(bottleneck, d), nn.GELU()
        nn.init.zeros_(self.up.weight); nn.init.zeros_(self.up.bias)
    def forward(self, x): return x + self.up(self.act(self.down(x)))

class Adapted(nn.Module):
    def __init__(self, inner, d, bottleneck): super().__init__(); self.inner, self.adapter = inner, Adapter(d, bottleneck)
    def forward(self, *a, **k): return self.adapter(self.inner(*a, **k))

def _swap(root, names, make):
    n = 0
    for _, mod in list(root.named_modules()):
        for cname, child in list(mod.named_children()):
            if isinstance(child, nn.Linear) and cname in names: setattr(mod, cname, make(child)); n += 1
    return n

def apply_method(model, method):
    bb, P = model.backbone, PEFT
    if method in ("full", "kd"):
        for p in model.parameters(): p.requires_grad = True
    else:
        for p in bb.parameters(): p.requires_grad = False
        if method == "lora": n = _swap(bb, P["targets"], lambda b: LoRALinear(b, P["r"], P["alpha"]))
        elif method == "lorma": n = _swap(bb, P["lorma_targets"], lambda b: LoRMALinear(b, P["r"], P["alpha"]))
        elif method == "qlora": n = _swap(bb, P["targets"], lambda b: QLoRALinear(b, P["r"], P["alpha"]))
        elif method == "adapter":
            n = 0
            for blk in getattr(bb, block_list_name(model.arch)):
                for cname, child in list(blk.named_children()):
                    if cname in ("attn", "self_attn", "ff"): setattr(blk, cname, Adapted(child, model.d_model, P["bottleneck"])); n += 1
        elif method == "bitfit":                                                                     
            n = 0
            for nm, p in bb.named_parameters():
                if nm.endswith(".bias"): p.requires_grad = True; n += 1
            for mod in bb.modules():
                if isinstance(mod, (nn.LayerNorm, RMSNorm)):
                    for p in mod.parameters(): p.requires_grad = True; n += 1
        elif method == "partial":                                                                     
            blocks = list(getattr(bb, block_list_name(model.arch))); n = 0
            for blk in blocks[-P["partial_last"]:]:
                for p in blk.parameters(): p.requires_grad = True; n += 1
            for nm, p in bb.named_parameters():
                if nm.split(".")[0] in ("ln_f", "norm_f", "enc_ln_f"): p.requires_grad = True; n += 1
        else: raise ValueError(f"unknown method {method}")
        assert n > 0, f"{method}: nothing to adapt"
    for p in list(model.head.parameters()) + list(model.att.parameters()): p.requires_grad = True
    tr = sum(p.numel() for p in model.parameters() if p.requires_grad); tot = sum(p.numel() for p in model.parameters())
    print(f"[{method}] trainable {tr/1e6:.3f}M of {tot/1e6:.2f}M ({100*tr/tot:.2f}%)")

def trainable_state(model):
    names = {n for n, p in model.named_parameters() if p.requires_grad}
    return {k: v.detach().cpu() for k, v in model.state_dict().items() if k in names}

In [39]:
def ml_metrics(Y, P):
    """Macro-F1 over ALL label columns (absent label -> 0, as sklearn zero_division=0), micro-F1, subset accuracy."""
    tp = (Y * P).sum(0); fp = ((1 - Y) * P).sum(0); fn = (Y * (1 - P)).sum(0); den = 2 * tp + fp + fn
    f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0)
    TP, FP, FN = tp.sum(), fp.sum(), fn.sum()
    return float(f1.mean()), float(2 * TP / max(2 * TP + FP + FN, 1)), float((Y == P).all(1).mean())

def decode(prob, th, ensure_one=None):
    P = (prob >= th).astype(np.int64)
    if ENSURE_ONE if ensure_one is None else ensure_one:
        none = P.sum(1) == 0; P[none, prob[none].argmax(1)] = 1
    return P

def tune_global(prob, Y):
    best_f, best_t = -1.0, 0.5
    for t in THRESH_GRID:
        f = ml_metrics(Y, decode(prob, t))[0]
        if f > best_f + 1e-9: best_f, best_t = f, t
    return best_t, best_f

sig = lambda l: 1.0 / (1.0 + np.exp(-l))



def build_doc(arch, p1_path, method, keep_every=None):
    bb, cfg, max_len = load_phase1_backbone(arch, p1_path); strip_unused(bb, arch)
    if keep_every: prune_layers(bb, arch, list(range(0, len(getattr(bb, block_list_name(arch))), keep_every)))
    assert WINDOW <= max_len, f"WINDOW {WINDOW} exceeds the Phase-1 max length {max_len}"
    model = DocModel(arch, bb, cfg["d_model"], PAD_ID, NL, BIDIRECTIONAL, POOL, USE_POS_FEATURE); apply_method(model, method)
    return model.to(DEVICE)

@torch.no_grad()
def predict_logits(model, pk, bs=4):
    model.eval(); out = []
    for b in range(0, len(pk["X"]), bs):
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            lg = model(pk["X"][b:b + bs].to(DEVICE), pk["M"][b:b + bs].to(DEVICE), pk["P"][b:b + bs].to(DEVICE))
        out.append(lg.float().cpu())
    return torch.cat(out).numpy()

def loss_fn(lg, y, pw, t_lg=None):
    hard = F.binary_cross_entropy_with_logits(lg.float(), y, pos_weight=pw)
    if t_lg is None: return hard
    soft = F.binary_cross_entropy_with_logits(lg.float() / KD_T, torch.sigmoid(t_lg.float() / KD_T)) * KD_T * KD_T
    return KD_ALPHA * hard + (1 - KD_ALPHA) * soft

def fit(model, pk, eval_fn, ckdir, method, class_w, teacher=None, seed=0, epochs=None):
    os.makedirs(ckdir, exist_ok=True); last_p, best_p = f"{ckdir}/last.pt", f"{ckdir}/best.pt"
    dm = nn.DataParallel(model) if NUM_GPUS > 1 else model
    small = {id(p) for p in list(model.head.parameters()) + list(model.att.parameters())}
    tr = [p for p in model.parameters() if p.requires_grad]
    groups = [{"params": [p for p in tr if id(p) in small], "lr": HEAD_LR}, {"params": [p for p in tr if id(p) not in small], "lr": LR[method]}]
    opt = torch.optim.AdamW([g for g in groups if g["params"]], weight_decay=WEIGHT_DECAY)
    base = [g["lr"] for g in opt.param_groups]; scaler = torch.amp.GradScaler("cuda", enabled=DEVICE.type == "cuda")
    ds = torch.utils.data.TensorDataset(pk["X"], pk["M"], pk["P"], pk["y"])
    loader = torch.utils.data.DataLoader(ds, batch_size=BATCH_DOCS, shuffle=True, drop_last=len(ds) > BATCH_DOCS, generator=torch.Generator().manual_seed(seed))
    epochs = epochs or EPOCHS[method]; spe = max(len(loader) // GRAD_ACCUM, 1); total, warm = epochs * spe, max(int(WARMUP_FRAC * epochs * spe), 1)
    cw = class_w.to(DEVICE); start, best, hist, bad = 0, -1.0, [], 0
    if os.path.exists(last_p):
        ck = torch.load(last_p, map_location="cpu", weights_only=False)
        model.load_state_dict(ck["trainable"], strict=False); opt.load_state_dict(ck["opt"]); scaler.load_state_dict(ck["scaler"])
        start, best, hist, bad = ck["epoch"], ck["best"], ck["hist"], ck["bad"]; print(f"resumed at epoch {start}/{epochs}")
    step, t0 = start * spe, time.time()
    for ep in range(start, epochs):
        dm.train(); run = nb = 0; opt.zero_grad(set_to_none=True)
        for i, (x, m, p, y) in enumerate(loader):
            x, m, p, y = x.to(DEVICE), m.to(DEVICE), p.to(DEVICE), y.to(DEVICE)
            with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
                lg = dm(x, m, p); tl = None
                if teacher is not None:
                    with torch.no_grad(): tl = teacher(x, m, p)
                loss = loss_fn(lg, y, cw, tl) / GRAD_ACCUM
            scaler.scale(loss).backward(); run += loss.item() * GRAD_ACCUM; nb += 1
            if (i + 1) % GRAD_ACCUM == 0:
                f = (step + 1) / warm if step < warm else 0.1 + 0.9 * 0.5 * (1 + math.cos(math.pi * (step - warm) / max(total - warm, 1)))
                for g, b in zip(opt.param_groups, base): g["lr"] = b * f
                scaler.unscale_(opt); torch.nn.utils.clip_grad_norm_(tr, 1.0); scaler.step(opt); scaler.update(); opt.zero_grad(set_to_none=True); step += 1
                if step % 50 == 0: print(f"    step {step}/{total} | {(time.time() - t0) / 60:.1f} min | running loss {run / max(nb, 1):.4f}", flush=True)
        f1 = eval_fn(model); hist.append(dict(epoch=ep + 1, loss=run / max(nb, 1), dev_macro_f1=f1))
        print(f"  epoch {ep+1}/{epochs} | loss {run/max(nb,1):.4f} | dev macro-F1 {f1:.4f} | {(time.time()-t0)/60:.1f} min", flush=True)
        if f1 > best: best, bad = f1, 0; torch.save({"trainable": trainable_state(model)}, best_p + ".tmp"); os.replace(best_p + ".tmp", best_p)
        else: bad += 1
        torch.save({"trainable": trainable_state(model), "opt": opt.state_dict(), "scaler": scaler.state_dict(), "epoch": ep + 1, "best": best, "hist": hist, "bad": bad}, last_p + ".tmp"); os.replace(last_p + ".tmp", last_p)
        if bad >= PATIENCE: print("  early stop (patience)"); break
        if RUN_MAX_MIN and (time.time() - t0) / 60 > RUN_MAX_MIN: print("  stopping: time limit for this run"); break
    model.load_state_dict(torch.load(best_p, map_location="cpu", weights_only=False)["trainable"], strict=False)
    return hist, best

In [40]:
Y_DEV = multihot(DOCS["dev"])
def setup_arch(arch):
    global ARCH, WORK, P1_MODEL, wt, TOK_SHA, PAD_ID, PACKS
    ARCH = arch; WORK = os.path.join(OUT_ROOT, f"{EXP_TAG}_{arch}"); os.makedirs(WORK, exist_ok=True); PACKS = {}
    P1_MODEL, p1_tok = find_phase1(arch); wt = WordTok(BPE.load(p1_tok)); PAD_ID = wt.pad
    TOK_SHA = hashlib.sha256(open(p1_tok, "rb").read()).hexdigest(); print(f"\n##### {arch}: model {P1_MODEL} | tokenizer {TOK_SHA[:10]}")
def get_packs(mode, K):
    if (mode, K) not in PACKS: PACKS[(mode, K)] = {sp: pack_docs(DOCS[sp], wt, mode, K) for sp in DOCS}
    return PACKS[(mode, K)]

def write_submission(P, name, outdir):
    sub = [{"id": d["id"], "pred": [int(i) for i in np.nonzero(row)[0]]} for d, row in zip(DOCS["test"], P)]
    pj = f"{outdir}/prediction.json"; json.dump(sub, open(pj, "w")); zp = f"{OUT_ROOT}/{name}_submission.zip"
    with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(pj, arcname="prediction.json")
    return zp

def choose_plan(budget_s, s_train, s_inf, n_tr, n_dv, n_te, e_max):
    avail = budget_s - (n_dv + n_te) * s_inf; per_epoch_eval = n_dv * s_inf
    for e in range(e_max, 1, -1):
        if (avail - e * per_epoch_eval) / (e * s_train) >= n_tr: return e, n_tr
    n2 = int((avail - 2 * per_epoch_eval) / (2 * s_train))
    if n2 >= 0.3 * n_tr: return 2, min(n2, n_tr)
    return 1, int(max(min((avail - per_epoch_eval) / s_train, n_tr), 8))

def subset_train(tr, n):
    N = len(tr["y"])
    if n >= N: return tr
    y = tr["y"].numpy(); rarity = (y / np.maximum(y.sum(0), 1)).max(1); order = np.argsort(-rarity, kind="stable"); n_rare = int(0.35 * n)
    keep = np.sort(np.concatenate([order[:n_rare], np.random.RandomState(SEED).choice(order[n_rare:], n - n_rare, replace=False)]))
    keep = torch.from_numpy(keep); return {k: v[keep] for k, v in tr.items()}

def plan_run(model, pk, method, teacher, budget_s):
    dm = nn.DataParallel(model) if NUM_GPUS > 1 else model; tr, dv, te = pk["train"], pk["dev"], pk["test"]; n_tr, n_dv, n_te = len(tr["y"]), len(dv["y"]), len(te["y"]); sync = torch.cuda.synchronize if DEVICE.type == "cuda" else (lambda: None)
    dm.train(); ts = []
    for i in range(5):
        idx = torch.arange(i * BATCH_DOCS, (i + 1) * BATCH_DOCS) % n_tr; x, m, p = (tr[k][idx].to(DEVICE) for k in ("X", "M", "P")); sync(); t = time.time()
        with torch.autocast(device_type="cuda", enabled=DEVICE.type == "cuda"):
            lg = dm(x, m, p)
            if teacher is not None:
                with torch.no_grad(): teacher(x, m, p)
        lg.float().sum().backward(); model.zero_grad(set_to_none=True); sync(); ts.append(time.time() - t)
    s_train = 1.1 * float(np.mean(ts[1:])) / BATCH_DOCS; k = min(16, n_dv); sync(); t = time.time(); predict_logits(model, {kk: v[:k] for kk, v in dv.items()}); sync(); s_inf = 1.1 * (time.time() - t) / k
    e, n = choose_plan(budget_s, s_train, s_inf, n_tr, n_dv, n_te, EPOCHS[method]); est = (e * n * s_train + e * n_dv * s_inf + (n_dv + n_te) * s_inf) / 60
    print(f"  plan: {s_train:.3f} s/doc train, {s_inf:.3f} s/doc inference | budget {budget_s/60:.0f} min -> {e} epoch(s) x {n}/{n_tr} docs, estimated {est:.0f} min")
    if budget_s < 600: print("  WARNING: very small time budget for this run - results will be weak")
    return subset_train(tr, n), e

def run_one(spec):
    method = spec["method"]; name = spec.get("name", method); mode = spec.get("mode", DOC_MODE); K = spec.get("k", K_CHUNKS)
    outdir = f"{WORK}/{name}"; os.makedirs(outdir, exist_ok=True)
    if os.path.exists(f"{outdir}/done.json"):
        print(f"[{ARCH}/{name}] already done -> skipped"); zp = f"{OUT_ROOT}/{EXP_TAG}_{ARCH}_{name}_submission.zip"
        if not os.path.exists(zp) and os.path.exists(f"{outdir}/prediction.json"):
            with zipfile.ZipFile(zp, "w", zipfile.ZIP_DEFLATED) as z: z.write(f"{outdir}/prediction.json", arcname="prediction.json")
        return json.load(open(f"{outdir}/done.json"))
    if TIME_BUDGET_MIN and not SMOKE:
        left_min = ((TIME_BUDGET_MIN - RESERVE_MIN - OVERHEAD_MIN * RUNS_LEFT) * 60 - (time.time() - SESSION_T0)) / 60 / RUNS_LEFT
        if left_min < MIN_RUN_MIN:
            print(f"[{ARCH}/{name}] SKIPPED: only {max(left_min, 0):.0f} min of budget left for this run (minimum {MIN_RUN_MIN}). Raise TIME_BUDGET_MIN and re-run this cell; finished runs are skipped."); return None
    print(f"\n{'='*20} {ARCH} / {name}  (mode={mode}, K={K}) {'='*20}"); t_start = time.time()
    model = build_doc(ARCH, P1_MODEL, method, STUDENT_KEEP_EVERY if method == "kd" else None); pk = get_packs(mode, K)
    ytr = pk["train"]["y"].numpy(); pos = ytr.sum(0); neg = len(ytr) - pos
    w = np.where(pos > 0, np.clip(np.sqrt(neg / np.maximum(pos, 1)), 1.0, POS_WEIGHT_MAX), 1.0)                                
    teacher = None
    if method == "kd":
        cands = [m for m in (json.load(open(p)) for p in glob.glob(f"{WORK}/*/done.json")) if m["method"] != "kd" and m["mode"] == mode and m["k"] == K]
        assert cands, "kd needs a finished non-kd run of this model (same mode/k) earlier in its list"
        tm = max(cands, key=lambda m: m["dev_f1_raw"]); print(f"  teacher = {tm['name']} (dev macro-F1 {tm['dev_f1_raw']:.4f}); student keeps every {STUDENT_KEEP_EVERY}nd block")
        teacher = build_doc(ARCH, P1_MODEL, tm["method"])
        teacher.load_state_dict(torch.load(f"{WORK}/{tm['name']}/ckpt/best.pt", map_location="cpu", weights_only=False)["trainable"], strict=False)
        teacher.eval(); [p.requires_grad_(False) for p in teacher.parameters()]
    ev = lambda m: tune_global(sig(predict_logits(m, pk["dev"])), Y_DEV)[1]
    trn, ep_run = pk["train"], EPOCHS[method]
    if TIME_BUDGET_MIN and not SMOKE:
        budget_s = max((TIME_BUDGET_MIN - RESERVE_MIN - OVERHEAD_MIN * RUNS_LEFT) * 60 - (time.time() - SESSION_T0), 0) / RUNS_LEFT
        trn, ep_run = plan_run(model, pk, method, teacher, budget_s)
    hist, best = fit(model, trn, ev, f"{outdir}/ckpt", method, torch.tensor(w, dtype=torch.float32), teacher, epochs=ep_run)
    if DEVICE.type == "cuda": torch.cuda.synchronize(); torch.cuda.reset_peak_memory_stats(); base_mem = torch.cuda.memory_allocated()
    t_inf = time.time(); ld = predict_logits(model, pk["dev"])
    if DEVICE.type == "cuda": torch.cuda.synchronize()
    infer_s = (time.time() - t_inf) / len(DOCS["dev"]); infer_gib = (torch.cuda.max_memory_allocated() - base_mem) / 2**30 if DEVICE.type == "cuda" else 0.0
    weights_gib = sum(p.numel() * p.element_size() for p in model.parameters()) / 2**30
    lt = predict_logits(model, pk["test"]); np.save(f"{outdir}/lg_dev.npy", ld); np.save(f"{outdir}/lg_test.npy", lt)
    pd_, pt_ = sig(ld), sig(lt); th, tuned = tune_global(pd_, Y_DEV); macro, micro, subset = ml_metrics(Y_DEV, decode(pd_, th))
    Yd, Pd = Y_DEV, decode(pd_, th); tp = (Yd * Pd).sum(0); den = 2 * tp + ((1 - Yd) * Pd).sum(0) + (Yd * (1 - Pd)).sum(0)
    lab_f1 = np.where(den > 0, 2 * tp / np.maximum(den, 1), 0.0); ntr = pk["train"]["y"].numpy().sum(0); rare = (ntr > 0) & (ntr < 5)
    f1_fixed = ml_metrics(Yd, decode(pd_, 0.5, ensure_one=False))[0]; f1_top1 = ml_metrics(Yd, decode(pd_, 2.0))[0]                                                                                  
    meta = dict(task="phase5_lsi", name=name, arch=ARCH, method=method, mode=mode, k=K, tag=EXP_TAG, tokenizer_sha=TOK_SHA, doc_ids_dev=[d["id"] for d in DOCS["dev"]],
                doc_ids_test=[d["id"] for d in DOCS["test"]], dev_f1_raw=macro, dev_f1_tuned=macro, dev_micro_f1=micro, dev_subset_acc=subset, threshold=th, n_labels=NL, dev_f1_fixed_0_5=f1_fixed, dev_f1_top1_only=f1_top1, rare_label_f1=(float(lab_f1[rare].mean()) if rare.any() else None), per_label_f1=[round(float(x), 4) for x in lab_f1], dev_support=Yd.sum(0).tolist(), lr=LR[method],
                trainable_M=round(sum(p.numel() for p in model.parameters() if p.requires_grad) / 1e6, 3), total_M=round(sum(p.numel() for p in model.parameters()) / 1e6, 2),
                weights_gib=round(weights_gib, 3), infer_sec_per_doc=round(infer_s, 4), infer_extra_gib=round(infer_gib, 3), epochs_run=len(hist), epochs_planned=ep_run, train_docs_used=len(trn["y"]), minutes=round((time.time() - t_start) / 60, 1), smoke=SMOKE)
    json.dump(meta, open(f"{outdir}/meta.json", "w"), indent=1)
    zp = write_submission(decode(pt_, th), f"{EXP_TAG}_{ARCH}_{name}", outdir)
    print(f"  dev macro-F1 {macro:.4f} | micro-F1 {micro:.4f} | subset-acc {subset:.4f} | threshold {th}\n  threshold strategies: fixed-0.5 {f1_fixed:.4f} | top-1-only {f1_top1:.4f} | tuned-global {macro:.4f} | rare-statute F1 {(lab_f1[rare].mean() if rare.any() else float('nan')):.4f}\n  submission: {zp}")
    json.dump(meta, open(f"{outdir}/done.json", "w"))
    del model, teacher; gc.collect(); torch.cuda.empty_cache()
    return meta

def zip_dir(zpath, files):
    with zipfile.ZipFile(zpath, "w", zipfile.ZIP_STORED) as z:
        for f, arc in files:
            if os.path.exists(f): z.write(f, arc)
    print(f"{os.path.getsize(zpath)/1e6:.1f} MB  {zpath}")

RESULTS = []
setup_arch(ARCH)
RUN_LIST = RUNS or DEFAULT_RUNS[ARCH]
for i, s in enumerate(RUN_LIST):
    RUNS_LEFT = len(RUN_LIST) - i; res = run_one(s)
    if res: RESULTS.append(res)
done = [os.path.dirname(p) for p in glob.glob(f"{WORK}/*/done.json")]
zip_dir(f"{OUT_ROOT}/{EXP_TAG}_{ARCH}_bundle.zip", [(f"{d}/{n}", f"{os.path.basename(WORK)}/{os.path.basename(d)}/{n}") for d in done for n in ("lg_dev.npy", "lg_test.npy", "meta.json")])
print("\nSESSION SUMMARY"); [print(f"  {m['arch']:11s} {m['name']:8s} raw {m['dev_f1_raw']:.4f} | tuned {m['dev_f1_tuned']:.4f} | {m['minutes']} min") for m in RESULTS]
dirs = [os.path.join(OUT_ROOT, d) for d in os.listdir(OUT_ROOT) if d.startswith(f"{EXP_TAG}_") and os.path.isdir(os.path.join(OUT_ROOT, d))]
zip_dir(f"{OUT_ROOT}/{EXP_TAG}_ALL_outputs.zip", [(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), OUT_ROOT)) for d in dirs for dp, _, fs in os.walk(d) for f in fs if f != "last.pt"])
print("Download the *_submission.zip files and the ALL_outputs zip from the Output panel BEFORE closing. Do NOT use Save Version / Save & Run All.")


##### encdec: model /kaggle/input/datasets/kraditya344/encdec/encdec_model.pt | tokenizer 8623378cb3

==================== encdec / adapter  (mode=tail, K=4) ====================
[adapter] trainable 1.635M of 53.36M (3.06%)
  plan: 0.036 s/doc train, 0.016 s/doc inference | budget 133 min -> 4 epoch(s) x 42750/42750 docs, estimated 118 min
    step 50/10684 | 0.5 min | running loss 0.7247
    step 100/10684 | 0.9 min | running loss 0.6061
    step 150/10684 | 1.4 min | running loss 0.5539
    step 200/10684 | 1.8 min | running loss 0.5274
    step 250/10684 | 2.3 min | running loss 0.5144
    step 300/10684 | 2.7 min | running loss 0.5049
    step 350/10684 | 3.2 min | running loss 0.4977
    step 400/10684 | 3.6 min | running loss 0.4925
    step 450/10684 | 4.1 min | running loss 0.4888
    step 500/10684 | 4.5 min | running loss 0.4860
    step 550/10684 | 5.0 min | running loss 0.4838
    step 600/10684 | 5.5 min | running loss 0.4811
    step 650/10684 | 5.9 min | running loss 0.

In [41]:
                                                                                                                                
if ENSEMBLE:
    metas = []
    for root in (OUT_ROOT, INPUT_ROOT):
        for mp in glob.glob(os.path.join(root, "**", "meta.json"), recursive=True):
            try: m = json.load(open(mp))
            except Exception: continue
            if m.get("task") == "phase5_lsi" and not m.get("smoke") and os.path.exists(os.path.join(os.path.dirname(mp), "lg_dev.npy")):
                m["_dir"] = os.path.dirname(mp); metas.append(m)
    if ENSEMBLE_SCOPE == "arch": metas = [m for m in metas if m["arch"] == ARCH]
    ETAG = f"{EXP_TAG}_{ARCH}_ensemble" if ENSEMBLE_SCOPE == "arch" else f"{EXP_TAG}_ensemble"
    seen = {}
    for m in metas: seen.setdefault((m["arch"], m["name"]), m)
    metas = sorted(seen.values(), key=lambda m: -m["dev_f1_raw"]); assert metas, "no finished runs found"
    metas = [m for m in metas if m["doc_ids_dev"] == metas[0]["doc_ids_dev"] and m["doc_ids_test"] == metas[0]["doc_ids_test"]]
    LD = [np.load(m["_dir"] + "/lg_dev.npy") for m in metas]; LT = [np.load(m["_dir"] + "/lg_test.npy") for m in metas]
    print(f"{len(metas)} runs found:"); [print(f"  {m['arch']:11s} {m['name']:8s} dev macro-F1 {m['dev_f1_raw']:.4f}") for m in metas]
    PD, PT = [sig(x) for x in LD], [sig(x) for x in LT]
    def finish(idx, name):
        pd_, pt_ = np.mean([PD[i] for i in idx], 0), np.mean([PT[i] for i in idx], 0); th, _ = tune_global(pd_, Y_DEV)
        macro, micro, subset = ml_metrics(Y_DEV, decode(pd_, th))
        od = f"{OUT_ROOT}/{ETAG}_{name}"; os.makedirs(od, exist_ok=True); zp = write_submission(decode(pt_, th), f"{ETAG}_{name}", od)
        print(f"  {name:8s} members={len(idx)} dev macro-F1 {macro:.4f} | micro-F1 {micro:.4f} | subset-acc {subset:.4f} | threshold {th} -> {zp}")
    finish(list(range(len(metas))), "all"); finish(list(range(min(4, len(metas)))), "top4")
    sel = [0]; best = tune_global(PD[0], Y_DEV)[1]                                                                                        
    while True:
        gains = [(tune_global(np.mean([PD[j] for j in sel + [i]], 0), Y_DEV)[1], i) for i in range(len(metas)) if i not in sel]
        if not gains or max(gains)[0] <= best + 1e-4: break
        best, nxt = max(gains); sel.append(nxt)
    finish(sel, "greedy")

2 runs found:
  encdec      adapter  dev macro-F1 0.0746
  encdec      partial  dev macro-F1 0.0714
  all      members=2 dev macro-F1 0.0723 | micro-F1 0.0826 | subset-acc 0.0000 | threshold 0.075 -> /kaggle/working/p5_encdec_ensemble_all_submission.zip
  top4     members=2 dev macro-F1 0.0723 | micro-F1 0.0826 | subset-acc 0.0000 | threshold 0.075 -> /kaggle/working/p5_encdec_ensemble_top4_submission.zip
  greedy   members=1 dev macro-F1 0.0746 | micro-F1 0.0922 | subset-acc 0.0000 | threshold 0.075 -> /kaggle/working/p5_encdec_ensemble_greedy_submission.zip


In [42]:
import pandas as pd
from collections import Counter

ANALYSIS_DIR = f"{WORK}/analysis"; os.makedirs(ANALYSIS_DIR, exist_ok=True)
runs = [m for m in (json.load(open(p)) for p in sorted(glob.glob(f"{WORK}/*/meta.json"))) if not m.get("smoke")]
assert runs, "no finished (non-smoke) runs in this session"

cols = ("name", "method", "total_M", "trainable_M", "weights_gib", "infer_sec_per_doc", "infer_extra_gib", "minutes", "dev_f1_raw", "dev_micro_f1", "dev_subset_acc", "rare_label_f1")
cost = pd.DataFrame([{c: m.get(c) for c in cols} for m in runs]).sort_values("dev_f1_raw", ascending=False)
cost.to_csv(f"{ANALYSIS_DIR}/cost_and_quality.csv", index=False)
print("MODEL SIZE / MEMORY / INFERENCE COST (dev split, batch of 4 documents, one GPU)")
print(cost.round(4).to_string(index=False))

best = max(runs, key=lambda m: m["dev_f1_raw"]); print(f"\nerror analysis on: {ARCH}/{best['name']} (threshold {best['threshold']})")
Y = Y_DEV; P = decode(sig(np.load(f"{WORK}/{best['name']}/lg_dev.npy")), best["threshold"])
ntr = multihot(DOCS["train"]).sum(0); sup = Y.sum(0); tp = (Y * P).sum(0); fn = (Y * (1 - P)).sum(0); fp = ((1 - Y) * P).sum(0)
f1 = np.where(2 * tp + fp + fn > 0, 2 * tp / np.maximum(2 * tp + fp + fn, 1), 0.0)
per = pd.DataFrame(dict(statute=np.arange(NL), train_pos=ntr.astype(int), dev_support=sup.astype(int), tp=tp.astype(int), missed=fn.astype(int), false_pos=fp.astype(int), f1=f1.round(4)))
per.to_csv(f"{ANALYSIS_DIR}/per_statute.csv", index=False)
print("\nmost missed statutes:\n", per.sort_values("missed", ascending=False).head(10).to_string(index=False))
print("\nmost over-predicted statutes:\n", per.sort_values("false_pos", ascending=False).head(10).to_string(index=False))

pairs = Counter()
for y, p in zip(Y, P):
    for i in np.nonzero(y * (1 - p))[0]:
        for j in np.nonzero((1 - y) * p)[0]: pairs[(int(i), int(j))] += 1
top_pairs = [dict(missed_statute=i, predicted_instead=j, docs=c) for (i, j), c in pairs.most_common(15)]
print("\nmost confused pairs (gold statute missed while another statute was predicted):")
for r in top_pairs: print("  ", r)

by_gold = {}
for lo, hi, tag in ((1, 1, "1 gold statute"), (2, 3, "2-3 gold statutes"), (4, 10**6, "4+ gold statutes")):
    sel = (Y.sum(1) >= lo) & (Y.sum(1) <= hi)
    if sel.any():
        t, p_, g_ = (Y[sel] * P[sel]).sum(), P[sel].sum(), Y[sel].sum()
        by_gold[tag] = dict(docs=int(sel.sum()), precision=round(float(t / max(p_, 1)), 4), recall=round(float(t / max(g_, 1)), 4), mean_pred=round(float(P[sel].sum(1).mean()), 2), mean_gold=round(float(Y[sel].sum(1).mean()), 2))
print("\nmulti-label behaviour by number of gold statutes per document:"); [print(f"   {k}: {v}") for k, v in by_gold.items()]

co = Counter()
for d in DOCS["train"]:
    for a, b in ((x, y) for x in d["label"] for y in d["label"] if x < y): co[(a, b)] += 1
co_rows = []
for (a, b), c in co.most_common(15):
    both = (Y[:, a] == 1) & (Y[:, b] == 1)
    co_rows.append(dict(pair=(a, b), train_cooccur=c, dev_docs_with_both=int(both.sum()), both_predicted=int((P[both][:, a] * P[both][:, b]).sum()) if both.any() else None))
print("\nmost frequent co-occurring statute pairs (train) and whether the model recovers them on dev:"); [print("  ", r) for r in co_rows]

json.dump(dict(arch=ARCH, run=best["name"], confused_pairs=top_pairs, by_gold_count=by_gold, cooccurrence=co_rows), open(f"{ANALYSIS_DIR}/error_analysis.json", "w"), indent=1, default=str)
dirs = [os.path.join(OUT_ROOT, d) for d in os.listdir(OUT_ROOT) if d.startswith(f"{EXP_TAG}_") and os.path.isdir(os.path.join(OUT_ROOT, d))]
zip_dir(f"{OUT_ROOT}/{EXP_TAG}_ALL_outputs.zip", [(os.path.join(dp, f), os.path.relpath(os.path.join(dp, f), OUT_ROOT)) for d in dirs for dp, _, fs in os.walk(d) for f in fs if f != "last.pt"])

MODEL SIZE / MEMORY / INFERENCE COST (dev split, batch of 4 documents, one GPU)
   name  method  total_M  trainable_M  weights_gib  infer_sec_per_doc  infer_extra_gib  minutes  dev_f1_raw  dev_micro_f1  dev_subset_acc rare_label_f1
adapter adapter    53.36        1.635        0.199             0.0184            0.529    117.3      0.0746        0.0922             0.0          None
partial partial    52.17       28.787        0.194             0.0176            0.578     86.0      0.0714        0.0754             0.0          None

error analysis on: encdec/adapter (threshold 0.075)

most missed statutes:
  statute  train_pos  dev_support  tp  missed  false_pos     f1
      27        329           83   0      83          0 0.0000
      50        318           80   0      80          0 0.0000
      92        309           77   0      77          0 0.0000
      84        272           68   0      68          0 0.0000
      65        351          105  37      68       3208 0.0221
      25 